## Step 2  
Input: s3://thesis--ec331-s3/enriched-volume-bids/  
Output: s3://thesis--ec331-s3/melted-volume-bids/  

TODO: I need to recognise what has been loaded and not to avoid duplication

In [ ]:
# No manifest files
import pandas as pd
import awswrangler as wr
import time
from datetime import datetime
import gc
import os
import psutil
import boto3

def get_memory_usage():
    """Return the current memory usage of the process in GB"""
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024 / 1024 / 1024
    return memory_gb

print(f"Initial memory usage: {get_memory_usage():.2f} GB")

# List of band columns we want to melt
band_cols = [f"BANDAVAIL{i}" for i in range(1, 11)]

def melt_and_write_chunks(df, chunk_size=100000, timestamp=None):
    """
    Melts the BANDAVAIL columns in chunks and writes each chunk directly to S3
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        
    print(f"Starting streaming melt of dataframe with shape: {df.shape}")
    print(f"Current memory usage: {get_memory_usage():.2f} GB")
    start_time = time.time()
    
    # Calculate number of chunks
    num_chunks = (len(df) + chunk_size - 1) // chunk_size
    print(f"Processing in {num_chunks} chunks of size {chunk_size}")
    
    # Create output path
    output_base = f"s3://thesis--ec331-s3/melted-volume-bids/enriched_volume_bids_melted_{timestamp}"
    
    # Track total rows processed
    total_rows_processed = 0
    all_chunk_files = []
    
    # Process dataframe in chunks
    for i in range(num_chunks):
        chunk_start = i * chunk_size
        chunk_end = min((i + 1) * chunk_size, len(df))
        
        print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
        print(f"Memory before chunk processing: {get_memory_usage():.2f} GB")
        
        # Extract chunk and immediately release the original slice reference
        chunk = df.iloc[chunk_start:chunk_end].copy()
        
        # Identify columns not being melted
        id_vars_cols = [col for col in chunk.columns if col not in band_cols]
        
        # Keep only necessary columns
        chunk = chunk[id_vars_cols + [col for col in band_cols if col in chunk.columns]]
        
        # Melt this chunk
        chunk_start_time = time.time()
        chunk_melted = pd.melt(
            chunk,
            id_vars=id_vars_cols,
            value_vars=[col for col in band_cols if col in chunk.columns],
            var_name="BIDBAND",
            value_name="BIDVOLUME"
        )
        
        # Clean up original chunk to free memory
        del chunk
        gc.collect()
        
        # Extract the band number
        chunk_melted["BIDBAND"] = chunk_melted["BIDBAND"].str.extract(r"BANDAVAIL(\d+)").astype(int)
        
        # Filter out null values to reduce size
        chunk_melted = chunk_melted.dropna(subset=["BIDVOLUME"])
        
        # Count rows in this chunk
        chunk_rows = len(chunk_melted)
        total_rows_processed += chunk_rows
        
        # Write this chunk directly to S3
        chunk_output = f"{output_base}_part{i+1:04d}.parquet"
        write_start = time.time()
        
        try:
            wr.s3.to_parquet(
                df=chunk_melted,
                path=chunk_output,
                index=False,
                compression="snappy"
            )
            all_chunk_files.append(chunk_output)
            write_time = time.time() - write_start
            print(f"  ✓ Chunk {i+1} written to S3 in {write_time:.2f} seconds ({chunk_rows} rows)")
        except Exception as e:
            print(f"  ✗ Error writing chunk {i+1} to S3: {str(e)}")
        
        # Clean up melted chunk to free memory
        del chunk_melted
        gc.collect()
        
        print(f"  Memory after chunk processing: {get_memory_usage():.2f} GB")
        print(f"  Chunk {i+1} processed in {time.time() - chunk_start_time:.2f} seconds")
    
    total_time = time.time() - start_time
    print(f"All chunks processed and written in {total_time:.2f} seconds")
    print(f"Total rows processed: {total_rows_processed}")
    print(f"Final memory usage: {get_memory_usage():.2f} GB")
    
    return output_base, all_chunk_files, total_rows_processed

# Create timestamp for consistent file naming
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Locate all the enriched volume bids files in S3 using boto3 for better pagination
print("Identifying enriched volume bids files in S3...")
try:
    s3_client = boto3.client('s3')
    bucket = "thesis--ec331-s3"
    prefix = "enriched-volume-bids/"
    enriched_files = []
    
    # Use pagination to handle large number of files
    paginator = s3_client.get_paginator('list_objects_v2')
    page_iterator = paginator.paginate(Bucket=bucket, Prefix=prefix)
    
    for page in page_iterator:
        if 'Contents' in page:
            for obj in page['Contents']:
                file_path = f"s3://{bucket}/{obj['Key']}"
                enriched_files.append(file_path)
    
    print(f"Found {len(enriched_files)} enriched volume bids files")
    
    if not enriched_files:
        raise ValueError("No enriched volume bids files found in S3")
        
except Exception as e:
    print(f"Error listing enriched volume bids files: {str(e)}")
    raise

# Process the files in batches to avoid memory issues
batch_size = 10  # Number of files to process at once
num_batches = (len(enriched_files) + batch_size - 1) // batch_size

print(f"Will process files in {num_batches} batches of up to {batch_size} files each")

# Create a list to store paths of all melted files
all_melted_files = []
global_total_rows = 0  # To accumulate total rows processed across batches

# Process each batch
for batch_num in range(num_batches):
    batch_start = batch_num * batch_size
    batch_end = min((batch_num + 1) * batch_size, len(enriched_files))
    batch_files = enriched_files[batch_start:batch_end]
    
    print(f"\nProcessing batch {batch_num + 1}/{num_batches} with {len(batch_files)} files")
    print(f"Memory before batch processing: {get_memory_usage():.2f} GB")
    
    try:
        # Read this batch of files
        print(f"Reading batch of {len(batch_files)} files...")
        batch_start_time = time.time()
        
        # Read the batch of parquet files using dataset=False because batch_files is a list
        batch_df = wr.s3.read_parquet(
            path=batch_files,
            dataset=False
        )
        
        print(f"Read batch in {time.time() - batch_start_time:.2f} seconds")
        print(f"Batch data shape: {batch_df.shape}")
        print(f"Memory after reading batch: {get_memory_usage():.2f} GB")
        
        # Melt and write this batch
        batch_timestamp = f"{timestamp}_batch{batch_num+1:03d}"
        output_base, batch_melted_files, batch_rows = melt_and_write_chunks(batch_df, chunk_size=50000, timestamp=batch_timestamp)
        
        # Add these files to our master list and update the row count
        all_melted_files.extend(batch_melted_files)
        global_total_rows += batch_rows
        
        # Clean up to free memory
        del batch_df
        gc.collect()
        
    except Exception as e:
        print(f"Error processing batch {batch_num + 1}: {str(e)}")
    
    print(f"Completed batch {batch_num + 1}/{num_batches}")
    print(f"Memory after batch processing: {get_memory_usage():.2f} GB")

print("\nAll batches processed!")
print(f"Total melted files created: {len(all_melted_files)}")

# Now, try to create a final DataFrame with all the melted data
print("\nLoading all melted data into final_df...")
try:
    # First, check if it's feasible to load all data at once
    est_rows_per_file = global_total_rows / len(all_melted_files) if all_melted_files else 0
    est_total_rows = est_rows_per_file * len(all_melted_files)
    
    print(f"Estimated total rows in all melted files: {est_total_rows:,.0f}")
    
    # Alternative: Load a sample first to estimate memory requirements
    sample_size = min(5, len(all_melted_files))
    if sample_size > 0:
        print(f"Loading sample of {sample_size} files to estimate memory requirements...")
        sample_df = wr.s3.read_parquet(path=all_melted_files[:sample_size], dataset=False)
        bytes_per_row = sample_df.memory_usage(deep=True).sum() / len(sample_df)
        est_memory_gb = (bytes_per_row * est_total_rows) / 1e9
        
        print(f"Sample loaded: {len(sample_df)} rows")
        print(f"Estimated memory required for full dataset: {est_memory_gb:.2f} GB")
        
        # Show sample data
        print("\nSample data preview:")
        print(sample_df.head())
        
        # Clean up sample
        del sample_df
        gc.collect()
    
    # Load all the data if it seems feasible or force loading
    force_load = True  # Set to True to load everything even if it might be large
    
    if force_load or (est_memory_gb < get_memory_usage() * 5):  # If estimated size is reasonable
        print("\nLoading all melted data...")
        final_df = wr.s3.read_parquet(path=all_melted_files, dataset=False)
        print(f"Successfully loaded all data into final_df")
        print(f"Final dataframe shape: {final_df.shape}")
        print(f"Final dataframe columns: {final_df.columns.tolist()}")
        print(f"Final dataframe preview:\n{final_df.head()}")
        
        # Optional: Save the final DataFrame to a single file for easier access later
        final_output_path = f"s3://thesis--ec331-s3/melted-volume-bids/combined_melted_data_{timestamp}.parquet"
        print(f"\nSaving final_df to {final_output_path}...")
        wr.s3.to_parquet(
            df=final_df,
            path=final_output_path,
            index=False,
            compression="snappy"
        )
        print(f"Successfully saved final_df to {final_output_path}")
    else:
        print(f"\nWarning: Full dataset would require approximately {est_memory_gb:.2f} GB of memory.")
        print("To load all data, set force_load = True in the code.")
        print("You can still access all the individual melted files from the all_melted_files list.")
        
except Exception as e:
    print(f"Error loading melted data: {str(e)}")
    print("The melted data is still available in the individual files.")
    print("You can find all file paths in the all_melted_files list.")

print("\nProcess complete!")
print(f"Final memory usage: {get_memory_usage():.2f} GB")

In [ ]:
import pandas as pd
import awswrangler as wr
import time
from datetime import datetime
import gc
import os
import psutil
import boto3

def get_memory_usage():
    """Return the current memory usage of the process in GB"""
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024 / 1024 / 1024
    return memory_gb

print(f"Initial memory usage: {get_memory_usage():.2f} GB")

# List of band columns we want to melt
band_cols = [f"BANDAVAIL{i}" for i in range(1, 11)]

def melt_and_write_chunks(df, chunk_size=100000, timestamp=None):
    """
    Melts the BANDAVAIL columns in chunks and writes each chunk directly to S3
    """
    if timestamp is None:
        timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
        
    print(f"Starting streaming melt of dataframe with shape: {df.shape}")
    print(f"Current memory usage: {get_memory_usage():.2f} GB")
    start_time = time.time()
    
    # Calculate number of chunks
    num_chunks = (len(df) + chunk_size - 1) // chunk_size
    print(f"Processing in {num_chunks} chunks of size {chunk_size}")
    
    # Create output path
    output_base = f"s3://thesis--ec331-s3/melted-volume-bids/enriched_volume_bids_melted_{timestamp}"
    
    # Track total rows processed
    total_rows_processed = 0
    all_chunk_files = []
    
    # Process dataframe in chunks
    for i in range(num_chunks):
        chunk_start = i * chunk_size
        chunk_end = min((i + 1) * chunk_size, len(df))
        
        print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
        print(f"Memory before chunk processing: {get_memory_usage():.2f} GB")
        
        # Extract chunk and immediately release the original slice reference
        chunk = df.iloc[chunk_start:chunk_end].copy()
        
        # Identify columns not being melted
        id_vars_cols = [col for col in chunk.columns if col not in band_cols]
        
        # Keep only necessary columns
        chunk = chunk[id_vars_cols + [col for col in band_cols if col in chunk.columns]]
        
        # Melt this chunk
        chunk_start_time = time.time()
        chunk_melted = pd.melt(
            chunk,
            id_vars=id_vars_cols,
            value_vars=[col for col in band_cols if col in chunk.columns],
            var_name="BIDBAND",
            value_name="BIDVOLUME"
        )
        
        # Clean up original chunk to free memory
        del chunk
        gc.collect()
        
        # Extract the band number
        chunk_melted["BIDBAND"] = chunk_melted["BIDBAND"].str.extract(r"BANDAVAIL(\d+)").astype(int)
        
        # Filter out null values to reduce size
        chunk_melted = chunk_melted.dropna(subset=["BIDVOLUME"])
        
        # Count rows in this chunk
        chunk_rows = len(chunk_melted)
        total_rows_processed += chunk_rows
        
        # Write this chunk directly to S3
        chunk_output = f"{output_base}_part{i+1:04d}.parquet"
        write_start = time.time()
        
        try:
            wr.s3.to_parquet(
                df=chunk_melted,
                path=chunk_output,
                index=False,
                compression="snappy"
            )
            all_chunk_files.append(chunk_output)
            write_time = time.time() - write_start
            print(f"  ✓ Chunk {i+1} written to S3 in {write_time:.2f} seconds ({chunk_rows} rows)")
        except Exception as e:
            print(f"  ✗ Error writing chunk {i+1} to S3: {str(e)}")
        
        # Clean up melted chunk to free memory
        del chunk_melted
        gc.collect()
        
        print(f"  Memory after chunk processing: {get_memory_usage():.2f} GB")
        print(f"  Chunk {i+1} processed in {time.time() - chunk_start_time:.2f} seconds")
    
    total_time = time.time() - start_time
    print(f"All chunks processed and written in {total_time:.2f} seconds")
    print(f"Total rows processed: {total_rows_processed}")
    print(f"Final memory usage: {get_memory_usage():.2f} GB")
    
    return output_base, all_chunk_files, total_rows_processed

# Create timestamp for consistent file naming
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Locate all the enriched volume bids files in S3 using boto3 for better pagination
print("Identifying enriched volume bids files in S3...")
try:
    s3_client = boto3.client('s3')
    bucket = "thesis--ec331-s3"
    prefix = "enriched-volume-bids/"
    enriched_files = []
    
    # Use pagination to handle large number of files
    paginator = s3_client.get_paginator('list_objects_v2')
    page_iterator = paginator.paginate(Bucket=bucket, Prefix=prefix)
    
    for page in page_iterator:
        if 'Contents' in page:
            for obj in page['Contents']:
                file_path = f"s3://{bucket}/{obj['Key']}"
                enriched_files.append(file_path)
    
    print(f"Found {len(enriched_files)} enriched volume bids files")
    
    if not enriched_files:
        raise ValueError("No enriched volume bids files found in S3")
        
except Exception as e:
    print(f"Error listing enriched volume bids files: {str(e)}")
    raise

# Process the files in batches to avoid memory issues
batch_size = 10  # Number of files to process at once
num_batches = (len(enriched_files) + batch_size - 1) // batch_size

print(f"Will process files in {num_batches} batches of up to {batch_size} files each")

# Create a list to store paths of all melted file manifests and all melted files
melted_manifests = []
all_melted_files = []
global_total_rows = 0  # To accumulate total rows processed across batches

# Process each batch
for batch_num in range(num_batches):
    batch_start = batch_num * batch_size
    batch_end = min((batch_num + 1) * batch_size, len(enriched_files))
    batch_files = enriched_files[batch_start:batch_end]
    
    print(f"\nProcessing batch {batch_num + 1}/{num_batches} with {len(batch_files)} files")
    print(f"Memory before batch processing: {get_memory_usage():.2f} GB")
    
    try:
        # Read this batch of files
        print(f"Reading batch of {len(batch_files)} files...")
        batch_start_time = time.time()
        
        # Read the batch of parquet files using dataset=False because batch_files is a list
        batch_df = wr.s3.read_parquet(
            path=batch_files,
            dataset=False
        )
        
        print(f"Read batch in {time.time() - batch_start_time:.2f} seconds")
        print(f"Batch data shape: {batch_df.shape}")
        print(f"Memory after reading batch: {get_memory_usage():.2f} GB")
        
        # Melt and write this batch
        batch_timestamp = f"{timestamp}_batch{batch_num+1:03d}"
        output_base, batch_melted_files, batch_rows = melt_and_write_chunks(batch_df, chunk_size=50000, timestamp=batch_timestamp)
        
        # Add these files to our master list and update the row count
        all_melted_files.extend(batch_melted_files)
        global_total_rows += batch_rows
        
        # Clean up to free memory
        del batch_df
        gc.collect()
        
        # Create a manifest file for this batch
        try:
            manifest = pd.DataFrame({"file_path": batch_melted_files})
            manifest_path = f"{output_base}_manifest.csv"
            wr.s3.to_csv(manifest, manifest_path, index=False)
            print(f"Created manifest file for batch {batch_num + 1}: {manifest_path}")
            
            # Add this manifest to our list
            melted_manifests.append(manifest_path)
        except Exception as e:
            print(f"Error creating manifest file for batch {batch_num + 1}: {str(e)}")
        
    except Exception as e:
        print(f"Error processing batch {batch_num + 1}: {str(e)}")
    
    print(f"Completed batch {batch_num + 1}/{num_batches}")
    print(f"Memory after batch processing: {get_memory_usage():.2f} GB")

print("\nAll batches processed!")
print(f"Total melted files created: {len(all_melted_files)}")

# Create a master manifest of all manifests
try:
    if melted_manifests:
        master_manifest = pd.DataFrame({"manifest_path": melted_manifests})
        master_manifest_path = f"s3://thesis--ec331-s3/melted-volume-bids/master_manifest_{timestamp}.csv"
        wr.s3.to_csv(master_manifest, master_manifest_path, index=False)
        print(f"Created master manifest at: {master_manifest_path}")
except Exception as e:
    print(f"Error creating master manifest: {str(e)}")

# Now, try to create a final DataFrame with all the melted data
print("\nLoading all melted data into final_df...")
try:
    # First, check if it's feasible to load all data at once
    est_rows_per_file = global_total_rows / len(all_melted_files) if all_melted_files else 0
    est_total_rows = est_rows_per_file * len(all_melted_files)
    
    print(f"Estimated total rows in all melted files: {est_total_rows:,.0f}")
    
    # Alternative: Load a sample first to estimate memory requirements
    sample_size = min(5, len(all_melted_files))
    if sample_size > 0:
        print(f"Loading sample of {sample_size} files to estimate memory requirements...")
        sample_df = wr.s3.read_parquet(path=all_melted_files[:sample_size], dataset=False)
        bytes_per_row = sample_df.memory_usage(deep=True).sum() / len(sample_df)
        est_memory_gb = (bytes_per_row * est_total_rows) / 1e9
        
        print(f"Sample loaded: {len(sample_df)} rows")
        print(f"Estimated memory required for full dataset: {est_memory_gb:.2f} GB")
        
        # Show sample data
        print("\nSample data preview:")
        print(sample_df.head())
        
        # Clean up sample
        del sample_df
        gc.collect()
    
    # Load all the data if it seems feasible or force loading
    force_load = True  # Set to True to load everything even if it might be large
    
    if force_load or (est_memory_gb < get_memory_usage() * 5):  # If estimated size is reasonable
        print("\nLoading all melted data...")
        final_df = wr.s3.read_parquet(path=all_melted_files, dataset=False)
        print(f"Successfully loaded all data into final_df")
        print(f"Final dataframe shape: {final_df.shape}")
        print(f"Final dataframe columns: {final_df.columns.tolist()}")
        print(f"Final dataframe preview:\n{final_df.head()}")
        
        # Optional: Save the final DataFrame to a single file for easier access later
        final_output_path = f"s3://thesis--ec331-s3/melted-volume-bids/combined_melted_data_{timestamp}.parquet"
        print(f"\nSaving final_df to {final_output_path}...")
        wr.s3.to_parquet(
            df=final_df,
            path=final_output_path,
            index=False,
            compression="snappy"
        )
        print(f"Successfully saved final_df to {final_output_path}")
    else:
        print(f"\nWarning: Full dataset would require approximately {est_memory_gb:.2f} GB of memory.")
        print("To load all data, set force_load = True in the code.")
        print(f"You can still access all the individual melted files from {master_manifest_path}")
        
except Exception as e:
    print(f"Error loading melted data: {str(e)}")
    print("The melted data is still available in the individual files.")
    print(f"You can find all file paths in the master manifest: {master_manifest_path}")

print("\nProcess complete!")
print(f"Final memory usage: {get_memory_usage():.2f} GB")

Initial memory usage: 0.16 GB
Identifying enriched volume bids files in S3...
Found 2123 enriched volume bids files
Will process files in 213 batches of up to 10 files each

Processing batch 1/213 with 10 files
Memory before batch processing: 0.18 GB
Reading batch of 10 files...
Read batch in 0.76 seconds
Batch data shape: (100000, 47)
Memory after reading batch: 0.31 GB
Starting streaming melt of dataframe with shape: (100000, 47)
Current memory usage: 0.31 GB
Processing in 2 chunks of size 50000
Processing chunk 1/2 (rows 0 to 49999)
Memory before chunk processing: 0.31 GB
  ✓ Chunk 1 written to S3 in 1.39 seconds (500000 rows)
  Memory after chunk processing: 0.36 GB
  Chunk 1 processed in 3.60 seconds
Processing chunk 2/2 (rows 50000 to 99999)
Memory before chunk processing: 0.36 GB
  ✓ Chunk 2 written to S3 in 1.36 seconds (500000 rows)
  Memory after chunk processing: 0.36 GB
  Chunk 2 processed in 3.42 seconds
All chunks processed and written in 7.17 seconds
Total rows processed

In [4]:
import awswrangler as wr
files = wr.s3.list_objects("s3://thesis--ec331-s3/enriched-volume-bids/")
print(f"Found {len(files)} files")

Found 2123 files


In [ ]:
# Check for duplicate rows in volu
import boto3
import pandas as pd

# Initialize S3 client
s3 = boto3.client('s3')
bucket_name = "thesis--ec331-s3"
prefix = "melted-volume-bids/"

# List all parquet files in the directory
print("Listing parquet files...")
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
parquet_files = [obj['Key'] for obj in response.get('Contents', []) 
                if obj['Key'].endswith('.parquet')]

# Handle pagination if there are many files
while response.get('IsTruncated', False):
    response = s3.list_objects_v2(
        Bucket=bucket_name, 
        Prefix=prefix,
        ContinuationToken=response['NextContinuationToken']
    )
    parquet_files.extend([obj['Key'] for obj in response.get('Contents', []) 
                        if obj['Key'].endswith('.parquet')])

print(f"Found {len(parquet_files)} parquet files")

# Load all files
print("Loading files into DataFrame...")
dfs = []
for file_key in parquet_files:
    try:
        s3_uri = f"s3://{bucket_name}/{file_key}"
        df = pd.read_parquet(s3_uri)
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {file_key}: {e}")

# Concatenate all dataframes
print("Concatenating all DataFrames...")
if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)
    
    # Get initial info
    total_rows = len(combined_df)
    print(f"Total rows in combined DataFrame: {total_rows}")
    print(f"DataFrame shape: {combined_df.shape}")
    print(f"Memory usage: {combined_df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    
    # Check for duplicates
    print("Checking for duplicates...")
    duplicated = combined_df.duplicated()
    duplicate_count = duplicated.sum()
    duplicate_percentage = (duplicate_count / total_rows) * 100
    
    print(f"Found {duplicate_count} duplicate rows ({duplicate_percentage:.2f}%)")
    
    # Show examples of duplicated rows if there are any
    if duplicate_count > 0:
        print("\nExample of duplicated rows:")
        duplicate_examples = combined_df[duplicated].head(3)
        print(duplicate_examples)
    
    # Get deduplicated DataFrame if needed
    print("\nRemoving duplicates...")
    deduplicated_df = combined_df.drop_duplicates()
    print(f"DataFrame shape after deduplication: {deduplicated_df.shape}")
    
    # Optional: save deduplicated data
    # deduplicated_df.to_parquet("s3://thesis--ec331-s3/deduplicated-data.parquet")
else:
    print("No data found in the parquet files")

In [ ]:
# import pandas as pd
# import awswrangler as wr
# import time
# from datetime import datetime
# import gc
# import os
# import psutil

# # List of band columns we want to melt
# band_cols = [f"BANDAVAIL{i}" for i in range(1, 11)]

# def get_memory_usage():
#     """Return the current memory usage of the process in GB"""
#     process = psutil.Process(os.getpid())
#     memory_gb = process.memory_info().rss / 1024 / 1024 / 1024
#     return memory_gb

# def melt_and_write_chunks(df, chunk_size=50000):
#     """
#     Melts the BANDAVAIL columns in chunks and writes each chunk directly to S3
#     """

#     print(f"Starting streaming melt of dataframe with shape: {df.shape}")
#     print(f"Initial memory usage: {get_memory_usage():.2f} GB")
#     start_time = time.time()
    
#     # Calculate number of chunks
#     num_chunks = (len(df) + chunk_size - 1) // chunk_size
#     print(f"Processing in {num_chunks} chunks of size {chunk_size}")
    
#     # Create a timestamp for the file prefix
#     timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
#     output_base = f"s3://thesis--ec331-s3/melted-volume-bids/enriched_volume_bids_melted_{timestamp}"
    
#     # Track total rows processed
#     total_rows_processed = 0
    
#     # Process dataframe in chunks
#     for i in range(num_chunks):
#         chunk_start = i * chunk_size
#         chunk_end = min((i + 1) * chunk_size, len(df))
        
#         print(f"Processing chunk {i+1}/{num_chunks} (rows {chunk_start} to {chunk_end-1})")
#         print(f"Memory before chunk processing: {get_memory_usage():.2f} GB")
        
#         # Extract chunk and immediately release the original slice reference
#         chunk = df.iloc[chunk_start:chunk_end].copy()
        
#         # Identify columns not being melted
#         id_vars_cols = [col for col in chunk.columns if col not in band_cols]
        
#         # Keep only necessary columns
#         chunk = chunk[id_vars_cols + [col for col in band_cols if col in chunk.columns]]
        
#         # Melt this chunk
#         chunk_start_time = time.time()
#         chunk_melted = pd.melt(
#             chunk,
#             id_vars=id_vars_cols,
#             value_vars=[col for col in band_cols if col in chunk.columns],
#             var_name="BIDBAND",
#             value_name="BIDVOLUME"
#         )
        
#         # Clean up original chunk to free memory
#         del chunk
#         gc.collect()
        
#         # Extract the band number
#         chunk_melted["BIDBAND"] = chunk_melted["BIDBAND"].str.extract(r"BANDAVAIL(\d+)").astype(int)
        
#         # Filter out null values to reduce size
#         chunk_melted = chunk_melted.dropna(subset=["BIDVOLUME"])
        
#         # Count rows in this chunk
#         chunk_rows = len(chunk_melted)
#         total_rows_processed += chunk_rows
        
#         # Write this chunk directly to S3
#         chunk_output = f"{output_base}_part{i+1:04d}.parquet"
#         write_start = time.time()
        
#         try:
#             wr.s3.to_parquet(
#                 df=chunk_melted,
#                 path=chunk_output,
#                 index=False,
#                 compression="snappy"
#             )
#             write_time = time.time() - write_start
#             print(f"  ✓ Chunk {i+1} written to S3 in {write_time:.2f} seconds ({chunk_rows} rows)")
#         except Exception as e:
#             print(f"  ✗ Error writing chunk {i+1} to S3: {str(e)}")
        
#         # Clean up melted chunk to free memory
#         del chunk_melted
#         gc.collect()
        
#         print(f"  Memory after chunk processing: {get_memory_usage():.2f} GB")
#         print(f"  Chunk {i+1} processed in {time.time() - chunk_start_time:.2f} seconds")
    
#     total_time = time.time() - start_time
#     print(f"All chunks processed and written in {total_time:.2f} seconds")
#     print(f"Total rows processed: {total_rows_processed}")
#     print(f"Final memory usage: {get_memory_usage():.2f} GB")
    
#     return output_base

# # Now let's apply this to the enriched_df
# if 'enriched_df' in locals():
#     print("Streaming melt of the enriched dataframe...")
    
#     # Try to install psutil if not already installed
#     try:
#         import psutil
#     except ImportError:
#         print("Installing psutil package for memory monitoring...")
#         import sys
#         !{sys.executable} -m pip install psutil
#         import psutil
    
#     # Melt and write the dataframe using the streaming approach
#     output_base = melt_and_write_chunks(enriched_df, chunk_size=50000)
    
#     print(f"\nSummary:")
#     print(f"Successfully melted and wrote dataframe to S3")
#     print(f"Base path: {output_base}")
#     print(f"Check S3 console or use AWS CLI to view all files")
    
#     # Optional: Create a manifest file with the locations of all chunks
#     try:
#         # List all the chunks we created
#         pattern = f"{output_base}_part*.parquet"
#         chunk_files = wr.s3.list_objects(pattern)
        
#         # Write a simple manifest file
#         manifest = pd.DataFrame({"file_path": chunk_files})
#         manifest_path = f"{output_base}_manifest.csv"
#         wr.s3.to_csv(manifest, manifest_path, index=False)
#         print(f"Created manifest file with paths to all {len(chunk_files)} chunks: {manifest_path}")
#     except Exception as e:
#         print(f"Error creating manifest file: {str(e)}")
    
# else:
#     print("The enriched_df is not available. Please run the enrichment code first.")